<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_4/Zero_Few_CoT-Examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# Phi-3 Mini 4K Instruct — Оценка на датасете (качество + скорость)
# ================================================================

!pip install evaluate sacrebleu rouge-score -q

# ----------------------------------------------------------------
# 1. Импорты
# ----------------------------------------------------------------
import os
import gc
import re
import time
import torch
import warnings
from typing import List, Dict, Any

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForCausalLM,
    pipeline,
    GenerationConfig,
)

# Для метрик качества (попробуем импортировать, если есть)
try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False
    print("⚠️ Библиотека 'evaluate' не установлена. Метрики BLEU/ROUGE будут недоступны.")
    print("   Установите: pip install evaluate sacrebleu rouge-score")

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False
    print("⚠️ 'rouge-score' не установлена. ROUGE будет пропущен.")
    print("   Установите: pip install rouge-score")

# Подавляем предупреждение о GenerationMixin
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

# ----------------------------------------------------------------
# 2. Основные настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.2
TOP_P = 0.9
SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# 3. Проверка окружения
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print(f"PyTorch:       {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU:           {gpu_name}")
    print(f"VRAM:          {gpu_memory:.2f} GB")
    DTYPE = torch.float16
    print(f"Dtype:         {DTYPE}")
    torch.cuda.reset_peak_memory_stats()
else:
    print("⚠️ GPU не обнаружен. Модель будет загружена на CPU.")
    DTYPE = torch.float32

print("=" * 70)

# ----------------------------------------------------------------
# 4. Освобождение памяти
# ----------------------------------------------------------------
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# 5. Загрузка токенизатора и модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Токенизатор загружен")
print(f"Vocabulary size: {len(tokenizer):,}")
print(f"EOS token:       {tokenizer.eos_token!r}")
print(f"PAD token:       {tokenizer.pad_token!r}")

print("\n🔹 Загружаем конфигурацию модели...")
config = AutoConfig.from_pretrained(MODEL_NAME)
print("✅ Конфигурация загружена")

print("\n🔹 Загружаем модель:")
print(f"   {MODEL_NAME}")

model_kwargs = {
    "config": config,
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["device_map"] = "cpu"

start_load_time = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start_load_time

model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

# ----------------------------------------------------------------
# 6. Информация о модели
# ----------------------------------------------------------------
num_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров:      {num_parameters / 1e9:.3f} B")
print(f"Обучаемых параметров:  {trainable_parameters / 1e9:.3f} B")
print(f"Device map: {getattr(model, 'hf_device_map', 'N/A')}")
if torch.cuda.is_available():
    current_vram = torch.cuda.memory_allocated() / 1024**3
    peak_vram = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Занято VRAM:          {current_vram:.2f} GB")
    print(f"Пиковое VRAM:         {peak_vram:.2f} GB")
print("=" * 70)

# ----------------------------------------------------------------
# 7. Pipeline
# ----------------------------------------------------------------
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
)
print("\n✅ Pipeline успешно создан")

# ----------------------------------------------------------------
# 8. Функция генерации
# ----------------------------------------------------------------
def generate_text(
    prompt: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    do_sample: bool = True,
) -> Dict[str, Any]:
    """Генерирует ответ и возвращает текст + метрики производительности."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt должен быть непустой строкой")

    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template"):
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        formatted_prompt = prompt

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else 1.0,
        top_p=top_p if do_sample else 1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    start_time = time.time()
    with torch.inference_mode():
        result = generator(
            formatted_prompt,
            return_full_text=False,
            generation_config=gen_config,
            clean_up_tokenization_spaces=False,
        )
    end_time = time.time()

    generated_text = result[0]["generated_text"].strip() if result else ""
    generation_time = end_time - start_time

    input_tokens = len(tokenizer.encode(formatted_prompt))
    output_tokens = len(tokenizer.encode(generated_text))
    tokens_per_second = output_tokens / generation_time if generation_time > 0 else 0.0

    return {
        "text": generated_text,
        "metrics": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(generation_time, 3),
            "tokens_per_second": round(tokens_per_second, 2),
        }
    }

# ----------------------------------------------------------------
# 9. УЛУЧШЕННАЯ ФУНКЦИЯ ИЗВЛЕЧЕНИЯ ЧИСЛА
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """
    Извлекает число из текста, отдавая предпочтение числу после знака '=' или 'равно'.
    Если таких нет, возвращает последнее число в тексте.
    """
    text = text.replace(",", "")
    # Ищем число после '='
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            after_eq = parts[-1]
            nums = re.findall(r"[-+]?\d*\.?\d+", after_eq)
            if nums:
                return nums[0]
    # Ищем после 'равно'
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            after_eq = parts[-1]
            nums = re.findall(r"[-+]?\d*\.?\d+", after_eq)
            if nums:
                return nums[0]
    # Иначе все числа
    numbers = re.findall(r"[-+]?\d*\.?\d+", text)
    return numbers[-1] if numbers else ""

# ----------------------------------------------------------------
# 10. ДАТАСЕТ (расширенный)
# ----------------------------------------------------------------
dataset = [
    # Переводы (русский → английский)
    {"type": "translation", "input": "Переведи на английский: Привет, как дела?", "target": "Hello, how are you?"},
    {"type": "translation", "input": "Переведи на английский: Сегодня отличная погода.", "target": "Today is great weather."},
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    # Математика
    {"type": "math", "input": "Сколько будет 2 + 2? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 5 * 3? Ответ дай числом.", "target": "15"},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
]

print("\n" + "=" * 70)
print("ДАТАСЕТ ДЛЯ ОЦЕНКИ")
print("=" * 70)
for i, ex in enumerate(dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# 11. ЗАГРУЗКА МЕТРИК (если доступны)
# ----------------------------------------------------------------
bleu_metric = None
rouge_scorer_obj = None

if HAS_EVALUATE:
    try:
        bleu_metric = load("bleu")
    except Exception as e:
        print(f"⚠️ Не удалось загрузить BLEU: {e}")

if HAS_ROUGE:
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# ----------------------------------------------------------------
# 12. ФУНКЦИЯ ОЦЕНКИ
# ----------------------------------------------------------------
def evaluate_dataset(dataset: List[Dict]) -> Dict:
    """
    Прогоняет все примеры, собирает предсказания,
    вычисляет метрики качества и производительности.
    """
    predictions = []
    references = []
    example_types = []
    perf_metrics = []

    # Для накопления метрик качества
    bleu_scores = []
    rouge_scores = []
    math_correct = 0
    math_total = 0

    print("\n" + "=" * 70)
    print("ЗАПУСК ОЦЕНКИ НА ДАТАСЕТЕ")
    print("=" * 70)

    for idx, example in enumerate(dataset, 1):
        print(f"\n--- Пример {idx} ({example['type']}) ---")
        print(f"Вопрос: {example['input']}")
        print(f"Эталон: {example['target']}")

        # Генерация
        result = generate_text(example["input"], max_new_tokens=MAX_NEW_TOKENS)
        pred_text = result["text"]
        metrics = result["metrics"]

        print(f"Ответ модели: {pred_text}")

        # Сохраняем для метрик
        predictions.append(pred_text)
        references.append([example["target"]])   # для BLEU
        example_types.append(example["type"])
        perf_metrics.append(metrics)

        # Вывод скорости
        print(f"   ⏱ Время: {metrics['generation_time_sec']} сек, "
              f"токенов: {metrics['output_tokens']}, TPS: {metrics['tokens_per_second']}")

        # ----- Сбор метрик качества по типам -----
        if example["type"] == "translation":
            # BLEU (накапливаем для среднего)
            if bleu_metric is not None:
                try:
                    score = bleu_metric.compute(predictions=[pred_text], references=[[example["target"]]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                    print(f"   BLEU ошибка: {e}")

            # ROUGE-L
            if rouge_scorer_obj is not None:
                try:
                    scores = rouge_scorer_obj.score(example["target"], pred_text)
                    rouge_scores.append(scores['rougeL'].fmeasure)
                except Exception as e:
                    print(f"   ROUGE ошибка: {e}")

        elif example["type"] == "math":
            math_total += 1
            pred_num = extract_number(pred_text)
            ref_num = extract_number(example["target"])
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика: верно")
            else:
                print(f"   ❌ Математика: неверно (извлечено '{pred_num}', ожидалось '{ref_num}')")

    # ----- ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА -----
    print("\n" + "=" * 70)
    print("РЕЗУЛЬТАТЫ ОЦЕНКИ КАЧЕСТВА")
    print("=" * 70)

    # Средний BLEU по переводам
    if bleu_scores:
        avg_bleu = sum(bleu_scores) / len(bleu_scores)
        print(f"📊 Средний BLEU (переводы): {avg_bleu:.3f}")
    else:
        print("📊 BLEU: не доступен (установите evaluate и sacrebleu)")

    # Средний ROUGE-L
    if rouge_scores:
        avg_rouge = sum(rouge_scores) / len(rouge_scores)
        print(f"📊 Средний ROUGE-L (переводы): {avg_rouge:.3f}")
    else:
        print("📊 ROUGE-L: не доступен (установите rouge-score)")

    # Точность на математике
    if math_total > 0:
        accuracy = math_correct / math_total
        print(f"📊 Точность на математике: {accuracy:.2%} ({math_correct}/{math_total})")
    else:
        print("📊 Математических задач нет.")

    # ----- СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ -----
    print("\n" + "=" * 70)
    print("СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ")
    print("=" * 70)
    if perf_metrics:
        avg_input = sum(m["input_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_output = sum(m["output_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_time = sum(m["generation_time_sec"] for m in perf_metrics) / len(perf_metrics)
        avg_tps = sum(m["tokens_per_second"] for m in perf_metrics) / len(perf_metrics)
        print(f"Среднее токенов в запросе:      {avg_input:.1f}")
        print(f"Среднее сгенерировано токенов:  {avg_output:.1f}")
        print(f"Среднее время генерации:        {avg_time:.3f} сек.")
        print(f"Средняя скорость (TPS):         {avg_tps:.2f} токен/сек.")

    return {
        "predictions": predictions,
        "references": references,
        "perf_metrics": perf_metrics,
    }

# ----------------------------------------------------------------
# 13. ЗАПУСК
# ----------------------------------------------------------------
if __name__ == "__main__":
    results = evaluate_dataset(dataset)
    print("\n" + "=" * 70)
    print("✅ ОЦЕНКА ЗАВЕРШЕНА")
    print("=" * 70)

In [ ]:
# ================================================================
# Phi-3 Mini 4K Instruct — Few-shot (2 примера) с системной инструкцией
# ================================================================

!pip install evaluate sacrebleu rouge-score -q

import os, gc, re, time, torch, warnings
from typing import List, Dict, Any, Tuple
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForCausalLM,
    pipeline, GenerationConfig
)

# Подавляем предупреждения о clean_up_tokenization_spaces
warnings.filterwarnings("ignore", message=".*clean_up_tokenization_spaces.*")
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

# Импорт метрик
try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False
    print("⚠️ Установите: pip install evaluate sacrebleu")

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False
    print("⚠️ Установите: pip install rouge-score")

# ----------------------------------------------------------------
# Настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 64            # достаточно для коротких ответов
TEMPERATURE = 0.2
TOP_P = 0.9
SEED = 42
NUM_DEMOS = 2                  # количество демонстраций для каждого типа

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# Окружение
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DTYPE = torch.float16
    torch.cuda.reset_peak_memory_stats()
else:
    DTYPE = torch.float32
print("=" * 70)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# Загрузка модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Токенизатор загружен")

print("\n🔹 Загружаем модель...")
model_kwargs = {
    "config": AutoConfig.from_pretrained(MODEL_NAME),
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

start = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start
model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

# Информация о модели
print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.3f} B")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("=" * 70)

# Pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("\n✅ Pipeline создан")

# ----------------------------------------------------------------
# Вспомогательные функции
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """Извлекает последнее число или число после '=' / 'равно'."""
    text = text.replace(",", "")
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else ""

def clean_response(text: str) -> str:
    """Обрезает ответ при появлении нового вопроса или маркера."""
    markers = ["Вопрос:", "Сколько будет", "Переведи", "?"]
    for marker in markers:
        idx = text.find(marker)
        if idx != -1 and idx > 0:  # если маркер не в самом начале
            return text[:idx].strip()
    return text.strip()

# ----------------------------------------------------------------
# Демонстрации (по NUM_DEMOS примеров на тип)
# ----------------------------------------------------------------
DEMO_TRANSLATIONS: List[Tuple[str, str]] = [
    ("Переведи на английский: Привет, как дела?", "Hello, how are you?"),
    ("Переведи на английский: Сегодня отличная погода.", "Today is great weather."),
]

DEMO_MATH: List[Tuple[str, str]] = [
    ("Сколько будет 2 + 2? Ответ дай числом.", "4"),
    ("Сколько будет 5 * 3? Ответ дай числом.", "15"),
]

# ----------------------------------------------------------------
# Тестовый датасет (без демонстраций)
# ----------------------------------------------------------------
test_dataset = [
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
]

print("\n" + "=" * 70)
print("ТЕСТОВЫЙ ДАТАСЕТ")
print("=" * 70)
for i, ex in enumerate(test_dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# Генерация с Few-shot (2 примера) + системное сообщение
# ----------------------------------------------------------------
def generate_fewshot(
    question: str,
    demonstrations: List[Tuple[str, str]],
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> Dict[str, Any]:
    """
    Формирует диалог:
    - system: краткая инструкция (не задавай вопросов, отвечай кратко)
    - user/assistant пары демонстраций
    - user: текущий вопрос
    """
    # Системное сообщение (поддерживается в Phi-3)
    system_msg = "Ты — полезный ассистент. Отвечай кратко и только на последний вопрос. Не задавай новых вопросов."

    messages = [{"role": "system", "content": system_msg}]

    for inp, out in demonstrations:
        messages.append({"role": "user", "content": inp})
        messages.append({"role": "assistant", "content": out})

    messages.append({"role": "user", "content": question})

    # Применяем чат-шаблон
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()

    raw = result[0]["generated_text"].strip()
    # Пост-обработка
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw

    input_tokens = len(tokenizer.encode(formatted))
    output_tokens = len(tokenizer.encode(cleaned))
    gen_time = end_time - start_time
    tps = output_tokens / gen_time if gen_time > 0 else 0.0

    return {
        "text": cleaned,
        "metrics": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(gen_time, 3),
            "tokens_per_second": round(tps, 2),
        }
    }

# ----------------------------------------------------------------
# Загрузка метрик
# ----------------------------------------------------------------
bleu_metric = load("bleu") if HAS_EVALUATE else None
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True) if HAS_ROUGE else None

# ----------------------------------------------------------------
# Оценка
# ----------------------------------------------------------------
def evaluate_fewshot(test_dataset):
    predictions, refs, types, perf_metrics = [], [], [], []
    bleu_scores, rouge_scores = [], []
    math_correct, math_total = 0, 0

    print("\n" + "=" * 70)
    print("ЗАПУСК ОЦЕНКИ (FEW-SHOT, 2 демонстрации)")
    print("=" * 70)

    for idx, ex in enumerate(test_dataset, 1):
        ex_type = ex["type"]
        print(f"\n--- Пример {idx} ({ex_type}) ---")
        print(f"Вопрос: {ex['input']}")
        print(f"Эталон: {ex['target']}")

        demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
        result = generate_fewshot(ex["input"], demos)
        pred = result["text"]
        metrics = result["metrics"]

        print(f"Ответ модели: {pred}")
        print(f"   ⏱ {metrics['generation_time_sec']} сек, {metrics['output_tokens']} токенов, TPS: {metrics['tokens_per_second']}")

        predictions.append(pred)
        refs.append([ex["target"]])
        types.append(ex_type)
        perf_metrics.append(metrics)

        # Качество
        if ex_type == "translation":
            if bleu_metric:
                try:
                    score = bleu_metric.compute(predictions=[pred], references=[[ex["target"]]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                    pass
            if rouge_scorer_obj:
                try:
                    scores = rouge_scorer_obj.score(ex["target"], pred)
                    rouge_scores.append(scores['rougeL'].fmeasure)
                except:
                    pass
        elif ex_type == "math":
            math_total += 1
            pred_num = extract_number(pred)
            ref_num = extract_number(ex["target"])
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика верно")
            else:
                print(f"   ❌ Математика: извлечено '{pred_num}', ожидалось '{ref_num}'")

    # Итоги качества
    print("\n" + "=" * 70)
    print("РЕЗУЛЬТАТЫ КАЧЕСТВА")
    print("=" * 70)
    if bleu_scores:
        print(f"📊 Средний BLEU: {sum(bleu_scores)/len(bleu_scores):.3f}")
    else:
        print("📊 BLEU: не доступен (установите sacrebleu)")

    if rouge_scores:
        print(f"📊 Средний ROUGE-L: {sum(rouge_scores)/len(rouge_scores):.3f}")

    if math_total:
        print(f"📊 Точность на математике: {math_correct/math_total:.2%} ({math_correct}/{math_total})")

    # Средние метрики производительности
    print("\n" + "=" * 70)
    print("СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ")
    print("=" * 70)
    if perf_metrics:
        avg_in = sum(m["input_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_out = sum(m["output_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_time = sum(m["generation_time_sec"] for m in perf_metrics) / len(perf_metrics)
        avg_tps = sum(m["tokens_per_second"] for m in perf_metrics) / len(perf_metrics)
        print(f"Среднее токенов в запросе:   {avg_in:.1f}")
        print(f"Среднее сгенерировано токенов: {avg_out:.1f}")
        print(f"Среднее время генерации:     {avg_time:.3f} сек.")
        print(f"Средняя скорость (TPS):      {avg_tps:.2f} токен/сек.")

    return {"predictions": predictions, "perf_metrics": perf_metrics}

# ----------------------------------------------------------------
# Запуск
# ----------------------------------------------------------------
if __name__ == "__main__":
    results = evaluate_fewshot(test_dataset)
    print("\n" + "=" * 70)
    print("✅ ОЦЕНКА FEW-SHOT ЗАВЕРШЕНА")
    print("=" * 70)

In [ ]:
# ================================================================
# Phi-3 Mini 4K Instruct — Сравнение режимов: Few-shot, CoT, Self-Consistency
# Сложные математические демонстрации для CoT
# ================================================================

!pip install evaluate sacrebleu rouge-score -q

import os, gc, re, time, torch, warnings
from typing import List, Dict, Any, Tuple
from collections import Counter
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForCausalLM,
    pipeline, GenerationConfig
)

warnings.filterwarnings("ignore", message=".*clean_up_tokenization_spaces.*")
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False

# ----------------------------------------------------------------
# Настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 512            # для CoT нужно больше токенов
TEMPERATURE = 0.7
TOP_P = 0.9
SEED = 42
NUM_CONSISTENCY = 5             # количество генераций для Self-Consistency
USE_SYSTEM = True

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# Окружение
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DTYPE = torch.float16
    torch.cuda.reset_peak_memory_stats()
else:
    DTYPE = torch.float32
print("=" * 70)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# Загрузка модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Токенизатор загружен")

print("\n🔹 Загружаем модель...")
model_kwargs = {
    "config": AutoConfig.from_pretrained(MODEL_NAME),
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

start = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start
model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.3f} B")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("=" * 70)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("\n✅ Pipeline создан")

# ----------------------------------------------------------------
# Вспомогательные функции
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """Извлекает последнее число или число после '=' / 'равно'."""
    text = text.replace(",", "")
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else ""

def clean_response(text: str) -> str:
    """Обрезает ответ при появлении нового вопроса или маркера."""
    markers = ["Вопрос:", "Сколько будет", "Переведи", "?"]
    for marker in markers:
        idx = text.find(marker)
        if idx != -1 and idx > 0:
            return text[:idx].strip()
    return text.strip()

def extract_final_answer(text: str) -> str:
    """
    Извлекает финальный ответ из текста с рассуждениями.
    Ищет фразу 'Ответ:' или последнее число.
    """
    if "Ответ:" in text:
        parts = text.split("Ответ:")
        if len(parts) > 1:
            candidate = parts[-1].strip()
            nums = re.findall(r"[-+]?\d*\.?\d+", candidate)
            if nums:
                return nums[-1]
            return candidate
    return extract_number(text)

# ----------------------------------------------------------------
# Демонстрации
# ----------------------------------------------------------------
# Для перевода (одинаковы для всех режимов)
DEMO_TRANSLATIONS = [
    ("Переведи на английский: Привет, как дела?", "Hello, how are you?"),
    ("Переведи на английский: Сегодня отличная погода.", "Today is great weather."),
]

# Для математики (без CoT) – простые задачи
DEMO_MATH = [
    ("Сколько будет 2 + 2? Ответ дай числом.", "4"),
    ("Сколько будет 5 * 3? Ответ дай числом.", "15"),
]

# Для CoT – сложные многошаговые выражения с рассуждениями
DEMO_MATH_COT = [
    (
        "Вычисли значение выражения: (-80)-3*((-15)-4*(2-3*(12-5*(6-9)))). Ответ дай числом.",
        "Рассуждение:\n"
        "1. Внутренняя скобка: 6-9 = -3\n"
        "2. 5*(-3) = -15\n"
        "3. 12 - (-15) = 27\n"
        "4. 3*(27) = 81\n"
        "5. 2 - 81 = -79\n"
        "6. 4*(-79) = -316\n"
        "7. -15 - (-316) = 301\n"
        "8. 3*(301) = 903\n"
        "9. -80 - 903 = -983\n"
        "Ответ: -983."
    ),
    (
        "Вычисли значение выражения: (-210)+5*((-8)-2*(-10-3*(7-2*(4-11)))). Ответ дай числом.",
        "Рассуждение:\n"
        "1. Внутренняя скобка: 4-11 = -7\n"
        "2. 2*(-7) = -14\n"
        "3. 7 - (-14) = 21\n"
        "4. 3*(21) = 63\n"
        "5. -10 - 63 = -73\n"
        "6. 2*(-73) = -146\n"
        "7. -8 - (-146) = 138\n"
        "8. 5*138 = 690\n"
        "9. -210 + 690 = 480\n"
        "Ответ: 480."
    ),
]

# ----------------------------------------------------------------
# Тестовый датасет (включая сложное выражение)
# ----------------------------------------------------------------
test_dataset = [
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
    # Сложное выражение (аналогичное демонстрациям CoT)
    {"type": "math", "input": "Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом.", "target": "-830"},
]

print("\n" + "=" * 70)
print("ТЕСТОВЫЙ ДАТАСЕТ")
print("=" * 70)
for i, ex in enumerate(test_dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# Базовые функции генерации (без CoT)
# ----------------------------------------------------------------
def build_messages(demonstrations, question, system_msg=None):
    messages = []
    if system_msg and USE_SYSTEM:
        messages.append({"role": "system", "content": system_msg})
    for inp, out in demonstrations:
        messages.append({"role": "user", "content": inp})
        messages.append({"role": "assistant", "content": out})
    messages.append({"role": "user", "content": question})
    return messages

def generate_with_demos(question, demonstrations, system_msg=None,
                        max_new_tokens=MAX_NEW_TOKENS, temperature=0.2):
    messages = build_messages(demonstrations, question, system_msg)
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()
    raw = result[0]["generated_text"].strip()
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw
    return cleaned, end_time - start_time, formatted

# ----------------------------------------------------------------
# Генерация с CoT (пошаговое рассуждение)
# ----------------------------------------------------------------
def generate_cot(question, demonstrations, system_msg=None, max_new_tokens=MAX_NEW_TOKENS*2):
    messages = build_messages(demonstrations, question, system_msg)
    # Добавляем инструкцию к вопросу, если её нет
    if "шаг за шагом" not in question.lower():
        modified_question = question + " Давайте подумаем шаг за шагом."
        messages[-1]["content"] = modified_question
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()
    raw = result[0]["generated_text"].strip()
    # Не обрезаем по маркерам, чтобы не потерять рассуждения
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw
    return cleaned, end_time - start_time, formatted

# ----------------------------------------------------------------
# Self-Consistency (голосование)
# ----------------------------------------------------------------
def generate_consensus(question, demonstrations, system_msg=None,
                       n=NUM_CONSISTENCY, temperature=0.7):
    candidates = []
    for _ in range(n):
        answer, _, _ = generate_with_demos(
            question, demonstrations, system_msg,
            max_new_tokens=MAX_NEW_TOKENS, temperature=temperature
        )
        # Извлекаем число для математических задач
        if "сколько" in question.lower() or "вычисли" in question.lower() or "Ответ дай числом" in question:
            final = extract_number(answer)
        else:
            final = answer
        candidates.append(final)
    counter = Counter(candidates)
    most_common = counter.most_common(1)[0][0]
    return most_common, candidates

# ----------------------------------------------------------------
# Загрузка метрик
# ----------------------------------------------------------------
bleu_metric = load("bleu") if HAS_EVALUATE else None
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True) if HAS_ROUGE else None

# ----------------------------------------------------------------
# Функция оценки для заданного режима
# ----------------------------------------------------------------
def evaluate_mode(mode: str):
    print("\n" + "=" * 70)
    print(f"ОЦЕНКА В РЕЖИМЕ: {mode.upper()}")
    print("=" * 70)

    predictions = []
    refs = []
    perf_times = []
    perf_tokens = []
    math_correct = 0
    math_total = 0
    bleu_scores = []
    rouge_scores = []

    system_msg = "Ты — полезный ассистент. Отвечай кратко и только на последний вопрос. Не задавай новых вопросов."

    for idx, ex in enumerate(test_dataset, 1):
        ex_type = ex["type"]
        question = ex["input"]
        target = ex["target"]

        print(f"\n--- Пример {idx} ({ex_type}) ---")
        print(f"Вопрос: {question}")
        print(f"Эталон: {target}")

        if mode == "fewshot":
            demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
            answer, gen_time, _ = generate_with_demos(
                question, demos, system_msg, temperature=0.2
            )
        elif mode == "cot":
            if ex_type == "translation":
                demos = DEMO_TRANSLATIONS
                answer, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.2
                )
            else:
                demos = DEMO_MATH_COT
                answer, gen_time, _ = generate_cot(
                    question, demos, system_msg
                )
        elif mode == "selfconsistency":
            demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
            if ex_type == "translation":
                answer, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.2
                )
            else:
                answer, candidates = generate_consensus(
                    question, demos, system_msg, n=NUM_CONSISTENCY, temperature=0.7
                )
                # Оцениваем время как среднее по n генерациям (приблизительно)
                _, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.7
                )
                gen_time *= NUM_CONSISTENCY

        # Очистка ответа (для CoT не обрезаем полностью)
        if mode != "cot":
            answer = clean_response(answer)

        print(f"Ответ модели: {answer}")
        print(f"   ⏱ {gen_time:.3f} сек")

        predictions.append(answer)
        refs.append([target])
        perf_times.append(gen_time)
        perf_tokens.append(len(tokenizer.encode(answer)))

        # Метрики качества
        if ex_type == "translation":
            if bleu_metric:
                try:
                    score = bleu_metric.compute(predictions=[answer], references=[[target]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                      print(f"BLEU compute error: {e}")

            if rouge_scorer_obj:
                try:
                    rouge_scores.append(rouge_scorer_obj.score(target, answer)['rougeL'].fmeasure)
                except:
                    pass
        elif ex_type == "math":
            math_total += 1
            if mode == "selfconsistency":
                pred_num = answer  # уже извлечено число
            else:
                pred_num = extract_number(answer) if mode != "cot" else extract_final_answer(answer)
            ref_num = extract_number(target)
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика верно")
            else:
                print(f"   ❌ Математика: извлечено '{pred_num}', ожидалось '{ref_num}'")

    # Итоговые метрики
    print("\n" + "=" * 70)
    print(f"РЕЗУЛЬТАТЫ КАЧЕСТВА ({mode.upper()})")
    print("=" * 70)
    if bleu_scores:
        print(f"📊 Средний BLEU: {sum(bleu_scores)/len(bleu_scores):.3f}")
    else:
        print("📊 BLEU: не доступен")
    if rouge_scores:
        print(f"📊 Средний ROUGE-L: {sum(rouge_scores)/len(rouge_scores):.3f}")
    if math_total:
        print(f"📊 Точность на математике: {math_correct/math_total:.2%} ({math_correct}/{math_total})")

    print("\n" + "=" * 70)
    print(f"СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ ({mode.upper()})")
    print("=" * 70)
    if perf_times:
        avg_time = sum(perf_times)/len(perf_times)
        avg_tokens = sum(perf_tokens)/len(perf_tokens)
        avg_tps = avg_tokens / avg_time if avg_time > 0 else 0
        print(f"Среднее время генерации: {avg_time:.3f} сек.")
        print(f"Среднее число токенов в ответе: {avg_tokens:.1f}")
        print(f"Средняя скорость: {avg_tps:.2f} токен/сек.")

    return {
        "mode": mode,
        "predictions": predictions,
        "bleu": sum(bleu_scores)/len(bleu_scores) if bleu_scores else None,
        "rouge": sum(rouge_scores)/len(rouge_scores) if rouge_scores else None,
        "math_accuracy": math_correct/math_total if math_total else None,
        "avg_time": sum(perf_times)/len(perf_times) if perf_times else None,
    }

# ----------------------------------------------------------------
# Запуск всех режимов и сравнение
# ----------------------------------------------------------------
if __name__ == "__main__":
    modes = ["fewshot", "cot", "selfconsistency"]
    results = {}
    for mode in modes:
        results[mode] = evaluate_mode(mode)

    # Сравнительная таблица
    print("\n" + "=" * 70)
    print("СРАВНЕНИЕ РЕЖИМОВ")
    print("=" * 70)
    print(f"{'Режим':<18} {'BLEU':<8} {'ROUGE-L':<10} {'Math Acc':<10} {'Время, с':<10}")
    for mode, res in results.items():
        bleu = f"{res['bleu']:.3f}" if res['bleu'] is not None else "—"
        rouge = f"{res['rouge']:.3f}" if res['rouge'] is not None else "—"
        acc = f"{res['math_accuracy']:.2%}" if res['math_accuracy'] is not None else "—"
        t = f"{res['avg_time']:.3f}" if res['avg_time'] is not None else "—"
        print(f"{mode:<18} {bleu:<8} {rouge:<10} {acc:<10} {t:<10}")

    print("\n" + "=" * 70)
    print("✅ ВСЕ РЕЖИМЫ ОЦЕНЕНЫ")
    print("=" * 70)